In [10]:
import os
import time
import json
import requests
import pandas as pd

from bs4 import BeautifulSoup
from pathlib import Path
from difflib import SequenceMatcher
import re

ROOT = Path.cwd().parent
print(ROOT)

c:\Users\sebas\PycharmProjects\Git\Indiana-Jones-and-the-Domestic-Box-Office-Forecasting-Model


In [6]:
DATA_RAW = ROOT/"data/raw"

basics_path = DATA_RAW / "title.basics.tsv"
akas_path = DATA_RAW / "title.akas.tsv"

basics = pd.read_csv(
    basics_path,
    sep="\t",
    na_values="\\N",
    low_memory=False
)

akas = pd.read_csv(
    akas_path,
    sep="\t",
    na_values="\\N",
    low_memory=False
)

basics["startYear"] = pd.to_numeric(basics["startYear"], errors="coerce")

In [24]:
CACHE_DIR = ROOT/"data/cache/the_numbers"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

## Setup 'The Numbers' Scraping

In [59]:
def create_slug(title):
    title = str(title).strip()

    title = title.replace("&", "and")
    title = title.replace(".", "")
    title = title.replace(",", "")
    title = title.replace(":", "")
    title = title.replace("!", "")

    title = re.sub(r"[^a-zA-Z0-9]+", "-", title)
    title = re.sub(r"-+", "-", title)

    return title.strip("-")

In [60]:
def add_the_numbers_abbreviations(title):
    title = str(title)

    title = title.replace("Episode", "Ep")
    title = title.replace("episode", "Ep")

    return title

In [61]:
def move_leading_article_to_end(title):
    title = str(title).strip()

    articles = ["The", "A", "An"]

    for article in articles:
        prefix = article + " "
        if title.startswith(prefix):
            return title[len(prefix):] + " " + article

    return title

In [64]:
def make_title_variants(title):
    title = str(title).strip()

    variants = [
        title,
        move_leading_article_to_end(title),
    ]

    # Common The Numbers transformations
    variants.append(title.replace("'", "-"))              # Don't -> Don-t
    variants.append(title.replace(".", ""))               # A.D. -> AD
    variants.append(title.replace("Vol.", "Volume"))      # Vol. 2 -> Volume 2
    variants.append(title.replace("Vol", "Volume"))
    variants.append(title.split(":")[0])                  # Zathura: A Space Adventure -> Zathura
    variants.append(title.split("!")[0])                  # Yu-Gi-Oh!: The Movie -> Yu-Gi-Oh

    # Episode abbreviation
    variants.append(title.replace("Episode", "Ep"))

    # Apply article movement to all variants too
    more_variants = []

    for v in variants:
        more_variants.append(v)
        more_variants.append(move_leading_article_to_end(v))

    return list(dict.fromkeys(more_variants))

In [65]:
def generate_the_numbers_urls(title, year):
    title_variants = make_title_variants(title)

    slugs = list(dict.fromkeys([
        create_slug(t)
        for t in title_variants
    ]))

    urls = []

    for slug in slugs:
        urls.extend([
            f"https://www.the-numbers.com/movie/{slug}-({int(year)})",
            f"https://www.the-numbers.com/movie/{slug}"
        ])

    return list(dict.fromkeys(urls))

TESTING URL BUILDING FUNC.

In [66]:
print(generate_the_numbers_urls("Sinners", 2025))
print(generate_the_numbers_urls("Marty Supreme", 2025))
print(generate_the_numbers_urls("Spider-Man 2", 2004))
print(generate_the_numbers_urls("The SpongeBob Movie: Sponge Out of Water", 2015))
print(generate_the_numbers_urls("The Pirates! Band of Misfits", 2012))
print(generate_the_numbers_urls("A Man and a Woman", 2026))

['https://www.the-numbers.com/movie/Sinners-(2025)', 'https://www.the-numbers.com/movie/Sinners']
['https://www.the-numbers.com/movie/Marty-Supreme-(2025)', 'https://www.the-numbers.com/movie/Marty-Supreme']
['https://www.the-numbers.com/movie/Spider-Man-2-(2004)', 'https://www.the-numbers.com/movie/Spider-Man-2']
['https://www.the-numbers.com/movie/The-SpongeBob-Movie-Sponge-Out-of-Water-(2015)', 'https://www.the-numbers.com/movie/The-SpongeBob-Movie-Sponge-Out-of-Water', 'https://www.the-numbers.com/movie/SpongeBob-Movie-Sponge-Out-of-Water-The-(2015)', 'https://www.the-numbers.com/movie/SpongeBob-Movie-Sponge-Out-of-Water-The', 'https://www.the-numbers.com/movie/The-SpongeBob-Movie-(2015)', 'https://www.the-numbers.com/movie/The-SpongeBob-Movie', 'https://www.the-numbers.com/movie/SpongeBob-Movie-The-(2015)', 'https://www.the-numbers.com/movie/SpongeBob-Movie-The']
['https://www.the-numbers.com/movie/The-Pirates-Band-of-Misfits-(2012)', 'https://www.the-numbers.com/movie/The-Pirates

In [91]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

def find_working_movie_url_from_titles(
    primary_title,
    original_title,
    aka_titles,
    year,
    skip_primary=False
):
    candidate_titles = []

    if not skip_primary:
        candidate_titles.append(primary_title)

    candidate_titles.append(original_title)

    if isinstance(aka_titles, list):
        candidate_titles.extend(aka_titles)

    candidate_titles = [
        t for t in candidate_titles
        if pd.notna(t)
    ]

    candidate_titles = list(dict.fromkeys(candidate_titles))

    for title in candidate_titles:
        for url in generate_the_numbers_urls(title, year):
            try:
                response = requests.get(url, headers=headers, timeout=15)

                if response.status_code == 200 and "Movie Details" in response.text:
                    return url

                time.sleep(1)

            except Exception:
                continue

    return None

In [93]:
movies = basics[
    (basics["titleType"] == "movie") &
    (basics["startYear"].between(2004, 2024)) &
    (basics["isAdult"] == 0)
].copy()

us_titles = akas[akas["region"] == "US"].copy()

us_movies = movies.merge(
    us_titles[["titleId", "title", "region"]],
    left_on="tconst",
    right_on="titleId",
    how="inner"
)
us_movies = us_movies.drop(columns="genres", errors="ignore")

In [94]:
kaggle_df = pd.read_csv(
    DATA_RAW / "box_office_data_filtering(2000-2024).csv"
)

kaggle_df = kaggle_df.rename(columns={
    "Release Group": "kaggle_title",
    "Year": "kaggle_year",
    "Genres": "genres"
})

def normalize_title(title):
    if pd.isna(title):
        return None
    title = str(title).lower().strip()
    title = re.sub(r"[^a-z0-9\s]", "", title)
    title = re.sub(r"\s+", " ", title)
    return title

In [95]:
us_movies[us_movies["primaryTitle"]=="Yu-Gi-Oh!: The Movie"]

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,titleId,title,region
3365,tt0403703,movie,Yu-Gi-Oh!: The Movie,Yu-Gi-Oh! The Movie,0,2004.0,NaN,90,tt0403703,Yu-Gi-Oh!: The Movie,US


In [96]:
us_movies[us_movies["tconst"]=="tt0403703"]

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,titleId,title,region
3365,tt0403703,movie,Yu-Gi-Oh!: The Movie,Yu-Gi-Oh! The Movie,0,2004.0,NaN,90,tt0403703,Yu-Gi-Oh!: The Movie,US


In [40]:
us_movies["normalized_title"] = us_movies["primaryTitle"].apply(normalize_title)
kaggle_df["normalized_title"] = kaggle_df["kaggle_title"].apply(normalize_title)

us_movies["startYear"] = pd.to_numeric(us_movies["startYear"], errors="coerce")
kaggle_df["kaggle_year"] = pd.to_numeric(kaggle_df["kaggle_year"], errors="coerce")

theatrical_movies = us_movies.merge(
    kaggle_df[["normalized_title", "kaggle_year","genres"]],
    left_on=["normalized_title", "startYear"],
    right_on=["normalized_title", "kaggle_year"],
    how="inner"
)

theatrical_movies = theatrical_movies.drop_duplicates(
    subset="tconst"
).reset_index(drop=True)

print(theatrical_movies.shape)

theatrical_movies[[
    "tconst",
    "primaryTitle",
    "startYear",
    "genres"
]].head()

(3881, 14)


,tconst,primaryTitle,startYear,genres
0,tt0120667,Fantastic Four,2005.0,"Action, Fantasy, Science Fiction"
1,tt0121164,Corpse Bride,2005.0,"Romance, Fantasy, Animation"
2,tt0121766,Star Wars: Episode III - Revenge of the Sith,2005.0,"Adventure, Action, Science Fiction"
3,tt0167190,Hellboy,2004.0,"Fantasy, Action"
4,tt0167456,Thunderbirds,2004.0,"Action, Adventure, Comedy, Family"


In [83]:
title_candidates = (
    us_movies
    .groupby("tconst")
    .agg({
        "primaryTitle": "first",
        "originalTitle": "first",
        "startYear": "first",
        "title": lambda x: list(dict.fromkeys(x.dropna()))
    })
    .reset_index()
    .rename(columns={"title": "aka_titles"})
)

In [84]:
theatrical_movies = theatrical_movies.merge(
    title_candidates[["tconst", "aka_titles"]],
    on="tconst",
    how="left"
)

TESTING URL FINDING OF A SAMPLE OF THEATRICAL MOVIES

In [ ]:
test_urls = []

for _, row in theatrical_movies.head(10).iterrows():
    title = row["primaryTitle"]
    year = row["startYear"]

    url = find_working_movie_url_from_titles(
        primary_title=row["primaryTitle"],
        original_title=row["originalTitle"],
        aka_titles=row["aka_titles"],
        year=row["startYear"],
        skip_primary=False
    )
    
    test_urls.append({
        "tconst": row["tconst"],
        "title": title,
        "year": year,
        "the_numbers_url": url
    })

    print(title, "->", url)

test_urls_df = pd.DataFrame(test_urls)
test_urls_df

Fantastic Four -> https://www.the-numbers.com/movie/Fantastic-Four-(2005)
Corpse Bride -> https://www.the-numbers.com/movie/Corpse-Bride
Star Wars: Episode III - Revenge of the Sith -> https://www.the-numbers.com/movie/Star-Wars-Ep-III-Revenge-of-the-Sith
Hellboy -> https://www.the-numbers.com/movie/Hellboy
Thunderbirds -> https://www.the-numbers.com/movie/Thunderbirds
The Bank Job -> https://www.the-numbers.com/movie/Bank-Job-The
Exorcist: The Beginning -> https://www.the-numbers.com/movie/Exorcist-The-Beginning
Children of Men -> https://www.the-numbers.com/movie/Children-of-Men
Awake -> https://www.the-numbers.com/movie/Awake
2046 -> None


,tconst,title,year,the_numbers_url
0,tt0120667,Fantastic Four,2005.0,https://www.the-numbers.com/movie/Fantastic-Fo...
1,tt0121164,Corpse Bride,2005.0,https://www.the-numbers.com/movie/Corpse-Bride
2,tt0121766,Star Wars: Episode III - Revenge of the Sith,2005.0,https://www.the-numbers.com/movie/Star-Wars-Ep...
3,tt0167190,Hellboy,2004.0,https://www.the-numbers.com/movie/Hellboy
4,tt0167456,Thunderbirds,2004.0,https://www.the-numbers.com/movie/Thunderbirds
5,tt0200465,The Bank Job,2008.0,https://www.the-numbers.com/movie/Bank-Job-The
6,tt0204313,Exorcist: The Beginning,2004.0,https://www.the-numbers.com/movie/Exorcist-The...
7,tt0206634,Children of Men,2006.0,https://www.the-numbers.com/movie/Children-of-Men
8,tt0211933,Awake,2007.0,https://www.the-numbers.com/movie/Awake
9,tt0212712,2046,2004.0,NaN


In [ ]:
url_rows = []

for i, (_, row) in enumerate(theatrical_movies.iterrows(), start=1):
    title = row["primaryTitle"]
    year = row["startYear"]

    url = find_working_movie_url_from_titles(
        primary_title=row["primaryTitle"],
        original_title=row["originalTitle"],
        aka_titles=row["aka_titles"],
        year=row["startYear"],
        skip_primary=False
    )

    url_rows.append({
        "tconst": row["tconst"],
        "primaryTitle": title,
        "startYear": year,
        "the_numbers_url": url
    })

    if i % 100 == 0:
        print(f"Checked {i} movies...")

url_df = pd.DataFrame(url_rows)

url_df["found_url"] = url_df["the_numbers_url"].notna()

print(url_df["found_url"].mean())
print(url_df["found_url"].value_counts())

Checked 100 movies...
Checked 200 movies...
Checked 300 movies...
Checked 400 movies...
Checked 500 movies...
Checked 600 movies...
Checked 700 movies...
Checked 800 movies...
Checked 900 movies...
Checked 1000 movies...
Checked 1100 movies...
Checked 1200 movies...
Checked 1300 movies...
Checked 1400 movies...
Checked 1500 movies...
Checked 1600 movies...
Checked 1700 movies...
Checked 1800 movies...
Checked 1900 movies...
Checked 2000 movies...
Checked 2100 movies...
Checked 2200 movies...
Checked 2300 movies...
Checked 2400 movies...
Checked 2500 movies...
Checked 2600 movies...
Checked 2700 movies...
Checked 2800 movies...
Checked 2900 movies...
Checked 3000 movies...
Checked 3100 movies...
Checked 3200 movies...
Checked 3300 movies...
Checked 3400 movies...
Checked 3500 movies...
Checked 3600 movies...
Checked 3700 movies...
Checked 3800 movies...
0.6289616078330327
found_url
True     2441
False    1440
Name: count, dtype: int64


In [58]:
url_df.to_csv(
    ROOT/"data/processed/the_numbers_urls.csv",
    index=False
)

MAKE THE PROCESS ONE STEP WHEN I TURN THIS INTO A SCRIPT

In [86]:
failed_df = url_df[
    url_df["found_url"] == False
].copy()

print(failed_df.shape)

(1408, 6)


In [87]:
failed_df = failed_df.merge(
    theatrical_movies[["tconst", "originalTitle", "aka_titles"]],
    on="tconst",
    how="left"
)

In [100]:
import time

retry_rows = []

retry_found = 0

start_time = time.time()

for i, (_, row) in enumerate(failed_df.iterrows(), start=1):

    url = find_working_movie_url_from_titles(
        primary_title=row["primaryTitle"],
        original_title=row["originalTitle"],
        aka_titles=row["aka_titles"],
        year=row["startYear"],
        skip_primary=True
    )

    if url is not None:
        retry_found += 1

    retry_rows.append({
        "tconst": row["tconst"],
        "retry_url": url
    })

    if i % 50 == 0:

        elapsed_minutes = (time.time() - start_time) / 60

        retry_rate = retry_found / i

        print(
            f"[{i}/{len(failed_df)}] "
            f"Recovered: {retry_found} | "
            f"Retry hit rate: {retry_rate:.3f} | "
            f"Elapsed: {elapsed_minutes:.2f} min"
        )

[50/1408] Recovered: 11 | Retry hit rate: 0.220 | Elapsed: 5.51 min
[100/1408] Recovered: 18 | Retry hit rate: 0.180 | Elapsed: 10.87 min
[150/1408] Recovered: 28 | Retry hit rate: 0.187 | Elapsed: 16.78 min
[200/1408] Recovered: 30 | Retry hit rate: 0.150 | Elapsed: 23.12 min
[250/1408] Recovered: 36 | Retry hit rate: 0.144 | Elapsed: 28.99 min
[300/1408] Recovered: 37 | Retry hit rate: 0.123 | Elapsed: 34.66 min
[350/1408] Recovered: 42 | Retry hit rate: 0.120 | Elapsed: 41.30 min
[400/1408] Recovered: 46 | Retry hit rate: 0.115 | Elapsed: 47.44 min
[450/1408] Recovered: 46 | Retry hit rate: 0.102 | Elapsed: 53.36 min
[500/1408] Recovered: 50 | Retry hit rate: 0.100 | Elapsed: 59.39 min
[550/1408] Recovered: 53 | Retry hit rate: 0.096 | Elapsed: 65.32 min
[600/1408] Recovered: 59 | Retry hit rate: 0.098 | Elapsed: 70.24 min
[650/1408] Recovered: 65 | Retry hit rate: 0.100 | Elapsed: 75.69 min
[700/1408] Recovered: 72 | Retry hit rate: 0.103 | Elapsed: 81.12 min
[750/1408] Recovered: 

In [102]:
retry_df = pd.DataFrame(retry_rows)

url_df = url_df.merge(
    retry_df,
    on="tconst",
    how="left"
)

url_df["the_numbers_url"] = url_df["the_numbers_url"].fillna(url_df["retry_url"])
url_df["found_url"] = url_df["the_numbers_url"].notna()

print(url_df["found_url"].mean())
print(url_df["found_url"].value_counts())

0.667869105900541
found_url
True     2592
False    1289
Name: count, dtype: int64


In [108]:
url_df.to_csv(
    ROOT/"data/processed/the_numbers_urls.csv",
    index=False
)

In [107]:
url_df = url_df.drop(columns=["retry_url_y"])

In [109]:
kaggle_us = kaggle_df.copy()

# If you already renamed these earlier, this handles both cases
if "Release Group" in kaggle_us.columns:
    kaggle_us = kaggle_us.rename(columns={
        "Release Group": "kaggle_title",
        "Year": "kaggle_year"
    })

kaggle_us["kaggle_year"] = pd.to_numeric(
    kaggle_us["kaggle_year"],
    errors="coerce"
)

kaggle_us = kaggle_us[
    kaggle_us["Production_Countries"]
    .fillna("")
    .str.contains("United States of America", case=False, regex=False)
].copy()

print("Kaggle US-production movies:", kaggle_us.shape)

Kaggle US-production movies: (3153, 13)


In [110]:
url_check = url_df.copy()

url_check["normalized_title"] = url_check["primaryTitle"].apply(normalize_title)
kaggle_us["normalized_title"] = kaggle_us["kaggle_title"].apply(normalize_title)

url_check["startYear"] = pd.to_numeric(
    url_check["startYear"],
    errors="coerce"
)

In [111]:
url_us_check = url_check.merge(
    kaggle_us[[
        "normalized_title",
        "kaggle_year",
        "kaggle_title",
        "Production_Countries"
    ]],
    left_on=["normalized_title", "startYear"],
    right_on=["normalized_title", "kaggle_year"],
    how="inner"
)

url_us_check = url_us_check.drop_duplicates(
    subset="tconst"
).reset_index(drop=True)

In [112]:
print("Original URL hit rate:")
print(url_df["found_url"].mean())
print(url_df["found_url"].value_counts())

print("\nUS-production filtered URL hit rate:")
print(url_us_check["found_url"].mean())
print(url_us_check["found_url"].value_counts())

print("\nRows before:", len(url_df))
print("Rows after US-production filter:", len(url_us_check))

Original URL hit rate:
0.667869105900541
found_url
True     2592
False    1289
Name: count, dtype: int64

US-production filtered URL hit rate:
0.9201430274135876
found_url
True     2316
False     201
Name: count, dtype: int64

Rows before: 3881
Rows after US-production filter: 2517


In [113]:
scrape_ready_df = url_us_check[
    url_us_check["found_url"] == True
].copy()

print(scrape_ready_df.shape)

(2316, 9)


In [114]:
scrape_ready_df.to_csv(
    ROOT/"data/processed/scrape_ready_us_production_filtered.csv",
    index=False
)